In [1]:
import sys
import os
import glob

# 1. Khai báo đường dẫn SPARK_HOME
os.environ['SPARK_HOME'] = '/home/quang/spark-4.1.1-bin-hadoop3'

# 2. Thêm thư mục mã nguồn python của Spark vào hệ thống
sys.path.insert(0, '/home/quang/spark-4.1.1-bin-hadoop3/python')

# 3. TỰ ĐỘNG TÌM VÀ THÊM THƯ VIỆN PY4J
try:
    py4j_zip = glob.glob(os.path.join(os.environ['SPARK_HOME'], 'python/lib', 'py4j-*.zip'))[0]
    sys.path.insert(0, py4j_zip)
except IndexError:
    print("Cảnh báo: Không tìm thấy file py4j.zip trong thư mục spark/python/lib")

# 4. Bây giờ tiến hành khởi tạo Spark Session trực tiếp
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lab4_BigData") \
    .master("local[*]") \
    .getOrCreate()

# 5. Kiểm tra thành quả
print("Kết nối thành công! Phiên bản Spark của bạn là:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/26 17:31:09 WARN Utils: Your hostname, quang-VirtualBox, resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/05/26 17:31:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/26 17:31:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Kết nối thành công! Phiên bản Spark của bạn là: 4.1.1


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

## Câu 1

In [3]:
df_customer_list= spark.read.csv(
    "file:///media/sf_DS200_BigData/Lab4/Customer_List.csv", 
    header=True,       
    inferSchema=True,
    sep = ";"
)

df_customer_list.printSchema()

[Stage 1:===================>                                       (1 + 2) / 3]

root
 |-- Customer_Trx_ID: string (nullable = true)
 |-- Subscriber_ID: string (nullable = true)
 |-- Subscribe_Date: date (nullable = true)
 |-- First_Order_Date: date (nullable = true)
 |-- Customer_Postal_Code: string (nullable = true)
 |-- Customer_City: string (nullable = true)
 |-- Customer_Country: string (nullable = true)
 |-- Customer_Country_Code: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)



In [4]:
df_order_items= spark.read.csv(
    "file:///media/sf_DS200_BigData/Lab4/Order_Items.csv", 
    header=True,       
    inferSchema=True,
    sep = ";"
)

df_order_items.printSchema()

[Stage 3:===================>                                       (1 + 2) / 3]

root
 |-- Order_ID: string (nullable = true)
 |-- Order_Item_ID: integer (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Seller_ID: string (nullable = true)
 |-- Shipping_Limit_Date: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Freight_Value: double (nullable = true)



In [5]:
df_order_reviews= spark.read.csv(
    "file:///media/sf_DS200_BigData/Lab4/Order_Reviews.csv", 
    header=True,       
    inferSchema=True,
    sep = ";"
)

df_order_reviews.printSchema()

root
 |-- Review_ID: string (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Review_Score: string (nullable = true)
 |-- Review_Comment_Title_En: string (nullable = true)
 |-- Review_Comment_Message_En: string (nullable = true)
 |-- Review_Creation_Date: string (nullable = true)
 |-- Review_Answer_Timestamp: timestamp (nullable = true)



In [12]:
df_order_reviews.take(3)

[Row(Review_ID='7bc2406110b926393aa56f80a40eba40', Order_ID='73fc7af87114b39712e6da79b0a377eb', Review_Score='4', Review_Comment_Title_En=None, Review_Comment_Message_En=None, Review_Creation_Date='2024-01-18 00:00', Review_Answer_Timestamp=datetime.datetime(2024, 1, 18, 21, 46)),
 Row(Review_ID='80e641a11e56f04c1ad469d5645fdfde', Order_ID='a548910a1c6147796b98fdf73dbeba33', Review_Score='5', Review_Comment_Title_En=None, Review_Comment_Message_En=None, Review_Creation_Date='2024-03-10 00:00', Review_Answer_Timestamp=datetime.datetime(2024, 3, 11, 3, 5)),
 Row(Review_ID='228ce5500dc1d8e020d8d1322874b6f0', Order_ID='f9e4b658b201a9f2ecdecbb34bed034b', Review_Score='5', Review_Comment_Title_En=None, Review_Comment_Message_En=None, Review_Creation_Date='2024-02-17 00:00', Review_Answer_Timestamp=datetime.datetime(2024, 2, 18, 14, 36))]

In [6]:
df_orders= spark.read.csv(
    "file:///media/sf_DS200_BigData/Lab4/Orders.csv", 
    header=True,       
    inferSchema=True,
    sep = ";"
)

df_orders.printSchema()

root
 |-- Order_ID: string (nullable = true)
 |-- Customer_Trx_ID: string (nullable = true)
 |-- Order_Status: string (nullable = true)
 |-- Order_Purchase_Timestamp: timestamp (nullable = true)
 |-- Order_Approved_At: timestamp (nullable = true)
 |-- Order_Delivered_Carrier_Date: timestamp (nullable = true)
 |-- Order_Delivered_Customer_Date: timestamp (nullable = true)
 |-- Order_Estimated_Delivery_Date: timestamp (nullable = true)



In [7]:
df_products= spark.read.csv(
    "file:///media/sf_DS200_BigData/Lab4/Products.csv", 
    header=True,       
    inferSchema=True,
    sep = ";"
)

df_products.printSchema()

root
 |-- Product_ID: string (nullable = true)
 |-- Product_Category_Name: string (nullable = true)
 |-- Product_Weight_Gr: integer (nullable = true)
 |-- Product_Length_Cm: integer (nullable = true)
 |-- Product_Height_Cm: integer (nullable = true)
 |-- Product_Width_Cm: integer (nullable = true)



## Cau 2

In [8]:
total_orders = df_orders.select("Order_ID").distinct().count()
total_customers = df_customer_list.select("Subscriber_ID").distinct().count()
total_sellers = df_order_items.select("Seller_ID").distinct().count()

print(f"+ Tổng số đơn hàng: {total_orders:,}")
print(f"+ Số lượng khách hàng duy nhất: {total_customers:,}")
print(f"+ Số lượng người bán (sellers): {total_sellers:,}")

[Stage 22:===================>                                      (1 + 2) / 3]

+ Tổng số đơn hàng: 99,441
+ Số lượng khách hàng duy nhất: 99,382
+ Số lượng người bán (sellers): 3,095


In [9]:
df_customer_orders = df_orders.join(df_customer_list,"Customer_Trx_ID","inner")
orders_by_country = df_customer_orders.groupby("Customer_Country")\
                    .agg(F.countDistinct("Order_ID").alias("Total_Orders"))\
                    .orderBy(F.desc("Total_Orders"))
orders_by_country.show(truncate = False)
                             

[Stage 29:>                                                         (0 + 3) / 3]

+----------------+------------+
|Customer_Country|Total_Orders|
+----------------+------------+
|Germany         |41754       |
|France          |12848       |
|Netherlands     |11629       |
|Belgium         |5464        |
|Austria         |5043        |
|Switzerland     |3640        |
|United Kingdom  |3382        |
|Poland          |2139        |
|Czechia         |2034        |
|Italy           |2025        |
|Spain           |1651        |
|Portugal        |1336        |
|Sweden          |975         |
|Denmark         |905         |
|Serbia          |746         |
|Norway          |716         |
|Slovakia        |534         |
|Slovenia        |495         |
|Turkey          |485         |
|Greece          |412         |
+----------------+------------+
only showing top 20 rows


## Cau 4

In [10]:
df_orders_with_time = df_orders.withColumn("Order_Purchase_Timestamp", F.to_timestamp("Order_Purchase_Timestamp")) \
                               .withColumn("Year", F.year("Order_Purchase_Timestamp")) \
                               .withColumn("Month", F.month("Order_Purchase_Timestamp"))
orders_by_year_month = df_orders_with_time.groupBy("Year", "Month") \
    .agg(F.count("Order_ID").alias("Total_Orders")) \
    .orderBy(F.asc("Year"), F.desc("Month"))

orders_by_year_month.show(truncate=False)

[Stage 35:===================>                                      (1 + 2) / 3]

+----+-----+------------+
|Year|Month|Total_Orders|
+----+-----+------------+
|2022|12   |1           |
|2022|10   |324         |
|2022|9    |4           |
|2023|12   |5673        |
|2023|11   |7544        |
|2023|10   |4631        |
|2023|9    |4285        |
|2023|8    |4331        |
|2023|7    |4026        |
|2023|6    |3245        |
|2023|5    |3700        |
|2023|4    |2404        |
|2023|3    |2682        |
|2023|2    |1780        |
|2023|1    |800         |
|2024|10   |4           |
|2024|9    |16          |
|2024|8    |6512        |
|2024|7    |6292        |
|2024|6    |6167        |
+----+-----+------------+
only showing top 20 rows


## Cau 5

In [19]:
df_reviews_clean = df_order_reviews.withColumn(
    "Review_Score_Clean", 
    F.expr("try_cast(Review_Score as int)")
).filter(
    F.col("Review_Score_Clean").isNotNull() & 
    (F.col("Review_Score_Clean") >= 1) & 
    (F.col("Review_Score_Clean") <= 5)
)

overall_avg_score = df_reviews_clean.select(F.avg("Review_Score_Clean")).first()[0]

print(f"Điểm đánh giá trung bình toàn hệ thống: {overall_avg_score:.2f} / 5.0")
print("-" * 50)

[Stage 46:======================================>                   (2 + 1) / 3]

Điểm đánh giá trung bình toàn hệ thống: 4.09 / 5.0
--------------------------------------------------


## Cau 6

In [22]:
df_orders_2024 = df_orders.withColumn("Order_Purchase_Timestamp", F.to_timestamp("Order_Purchase_Timestamp")) \
                           .filter(F.year("Order_Purchase_Timestamp") == 2024)

df_sales_2024 = df_orders_2024 \
    .join(df_order_items, "Order_ID", "inner") \
    .join(df_products, "Product_ID", "inner")

df_revenue_by_cat = df_sales_2024 \
    .withColumn("Revenue", F.col("Price") + F.col("Freight_Value")) \
    .groupBy("Product_Category_Name") \
    .agg(F.sum("Revenue").alias("Total_Revenue")) \
    .orderBy(F.desc("Total_Revenue"))

df_revenue_by_cat.show(truncate=False)
### Tại sao cùng 1 order mà freight_value lại có thể khác nhau cho các sản phẩm khác nhau? 
### Nếu khác sp thì maybe vận chuyển khác nhưng cùng 1 sp thì freight tính n lần cho n quantity?
### Co thể seller_id khác nhau trong 1 đơn hàng phí ship phải tính lên
### Đơn hàng có thể trùng seller_id và 2 loại sp khác nhau thì freigh_value khác?
### Nếu ở shopee thì chung 1 seller_id thì phí ship tính 1 lần

[Stage 51:======================================>                   (2 + 1) / 3]

+-------------------------------+------------------+
|Product_Category_Name          |Total_Revenue     |
+-------------------------------+------------------+
|Health_Beauty                  |885191.1199999978 |
|Watches_Gifts                  |771986.7500000008 |
|Bed_Bath_Table                 |650794.7000000002 |
|Sports_Leisure                 |621999.3400000001 |
|Computers_Accessories          |594771.0399999998 |
|Housewares                     |491576.96000000084|
|Furniture_Decor                |476466.13000000024|
|Auto                           |404210.57000000024|
|Baby                           |299052.56000000006|
|Cool_Stuff                     |273910.05000000005|
|Garden_Tools                   |259068.31999999992|
|Telephony                      |217452.12999999977|
|Perfumery                      |204562.54000000015|
|Toys                           |200634.0700000001 |
|Office_Furniture               |181745.73000000016|
|Stationery                     |164743.85    

## Câu 7

In [24]:
df_product_sales = df_order_items.groupBy("Product_ID") \
    .agg(F.count("Order_Item_ID").alias("Sales_Volume"))

df_product_ratings = df_order_items.join(df_reviews_clean, "Order_ID", "inner") \
    .groupBy("Product_ID") \
    .agg(F.avg("Review_Score").alias("Avg_Review_Score"))
best_selling_products = df_product_sales.join(df_product_ratings, "Product_ID", "inner") \
    .orderBy(F.desc("Sales_Volume"))
best_selling_products.show(truncate=False)


[Stage 58:===================>                                      (1 + 2) / 3]

+--------------------------------+------------+------------------+
|Product_ID                      |Sales_Volume|Avg_Review_Score  |
+--------------------------------+------------+------------------+
|aca2eb7d00ea1a7b8ebd4e68314663af|527         |4.019083969465649 |
|99a4788cb24856965c36a24e339b6058|488         |3.8983402489626555|
|422879e10f46682990de24d770e7f83d|484         |3.9465020576131686|
|389d119b48cf3043d311335e499d9c6b|392         |4.117647058823529 |
|368c6c730842d78016ad823897a372db|388         |3.922680412371134 |
|53759a2ecddad2bb87a079a1f1519f73|373         |3.868632707774799 |
|d1c427060a0f73f6b889a5c7c61f2ac4|343         |4.194117647058824 |
|53b36df67ebb7c41585e8d54d6772e08|323         |4.190625          |
|154e7e31ebfa092203795c972e5804a6|281         |4.315068493150685 |
|3dd2a17168ec895c781a9191c1e95ad7|274         |4.209558823529412 |
|2b4609f8948be18874494203496bc318|260         |4.070631970260223 |
|7c1bd920dbdf22470b68bde975dd3ccf|231         |3.8765957446808

## Cau 8

In [26]:
df_delivery = df_orders.join(df_order_items, "Order_ID", "inner") \
    .withColumn("Order_Delivered_Carrier_Date", F.to_timestamp("Order_Delivered_Carrier_Date")) \
    .withColumn("Shipping_Limit_Date", F.to_timestamp("Shipping_Limit_Date"))

df_delivery_perf = df_delivery.withColumn(
    "Delivery_Delay_Days",
    F.datediff("Order_Delivered_Carrier_Date", "Shipping_Limit_Date")
).filter(F.col("Delivery_Delay_Days").isNotNull())

total_deliveries = df_delivery_perf.count()
on_time_deliveries = df_delivery_perf.filter(F.col("Delivery_Delay_Days") <= 0).count()
on_time_rate = (on_time_deliveries / total_deliveries) * 100
avg_delay = df_delivery_perf.select(F.avg("Delivery_Delay_Days")).first()[0]

print(f"+ Tổng số lượt giao hàng phân tích: {total_deliveries:,}")
print(f"+ Tỷ lệ giao hàng đúng hạn/sớm hạn: {on_time_rate:.2f}%")
print(f"+ Số ngày trễ trung bình của hệ thống: {avg_delay:.2f} ngày")
print("-" * 50)

df_delivery_perf.select("Order_ID", "Order_Delivered_Carrier_Date", "Shipping_Limit_Date", "Delivery_Delay_Days").show(10)

+ Tổng số lượt giao hàng phân tích: 111,456
+ Tỷ lệ giao hàng đúng hạn/sớm hạn: 94.02%
+ Số ngày trễ trung bình của hệ thống: -3.44 ngày
--------------------------------------------------
+--------------------+----------------------------+-------------------+-------------------+
|            Order_ID|Order_Delivered_Carrier_Date|Shipping_Limit_Date|Delivery_Delay_Days|
+--------------------+----------------------------+-------------------+-------------------+
|00010242fe8c5a6d1...|         2023-09-19 18:34:00|2023-09-19 09:45:00|                  0|
|00018f77f2f0320c5...|         2023-05-04 14:35:00|2023-05-03 11:05:00|                  1|
|000229ec398224ef6...|         2024-01-16 12:36:00|2024-01-18 14:48:00|                 -2|
|00024acbcdf0a6daa...|         2024-08-10 13:28:00|2024-08-15 10:10:00|                 -5|
|00042b26cf59d7ce6...|         2023-02-16 09:46:00|2023-02-13 13:57:00|                  3|
|00048cc3ae777c65d...|         2023-05-17 11:05:00|2023-05-23 03:55:00|     

## Cau 9

In [31]:
df_order_costs = df_customer_list \
    .join(df_orders, "Customer_Trx_ID", "inner") \
    .join(df_order_items, "Order_ID", "inner") \
    .withColumn("Item_Cost", F.col("Price") + F.col("Freight_Value")) \
    .groupBy("Subscriber_ID", "Order_ID") \
    .agg(F.sum("Item_Cost").alias("Order_Total_Value"))

df_customer_rfm = df_order_costs.groupBy("Subscriber_ID") \
    .agg(
        F.count("Order_ID").alias("Order_Count"),
        F.avg("Order_Total_Value").alias("Avg_Order_Value")
    )

df_customer_segments = df_customer_rfm.withColumn(
    "Customer_Segment",
    F.when(F.col("Order_Count") >= 5, "Loyal Customer")
     .when((F.col("Order_Count") >= 2) & (F.col("Order_Count") < 5), "Regular Customer")
     .otherwise("One-time Buyer")
)

df_segment_stats = df_customer_segments.groupBy("Customer_Segment") \
    .agg(
        F.count("Subscriber_ID").alias("Total_Customers"),
        F.avg("Avg_Order_Value").alias("Average_Spend_Per_Order")
    ).orderBy(F.desc("Total_Customers"))

df_segment_stats.show(truncate=False)

+----------------+---------------+-----------------------+
|Customer_Segment|Total_Customers|Average_Spend_Per_Order|
+----------------+---------------+-----------------------+
|One-time Buyer  |92507          |161.49163836249903     |
|Regular Customer|2894           |146.95751641326873     |
|Loyal Customer  |19             |131.0484799498747      |
+----------------+---------------+-----------------------+



## Cau 10

In [39]:
from pyspark.sql import Window
## Khong tinh gia ship vao doanh thu cua shop ???
df_seller_stats = df_order_items \
    .groupBy("Seller_ID") \
    .agg(
        F.sum("Price").alias("Total_Revenue"),
        F.count("Order_Item_ID").alias("Items_Sold")
    )

seller_window_revenue = Window.orderBy(F.desc("Total_Revenue"))
seller_window_items_sold = Window.orderBy(F.desc("Items_Sold"))
df_seller_ranked_revenue = df_seller_stats \
    .withColumn("Rank_By_Revenue", F.dense_rank().over(seller_window_revenue))
df_seller_ranked_items_sold = df_seller_stats \
    .withColumn("Rank_By_Items_Sold", F.dense_rank().over(seller_window_items_sold))
print("Xep hang Seller theo Revenue")
df_seller_ranked_revenue.show(15, truncate=False)
print("Xep hang Seller theo Items Sold")
df_seller_ranked_items_sold.show(15, truncate=False)

Xep hang Seller theo Revenue


26/05/26 18:07:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/26 18:07:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/26 18:07:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/26 18:08:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/26 18:08:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/26 18:08:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/26 1

+--------------------------------+------------------+----------+---------------+
|Seller_ID                       |Total_Revenue     |Items_Sold|Rank_By_Revenue|
+--------------------------------+------------------+----------+---------------+
|4869f7a5dfa277a7dca6462dcf3b52b2|229472.62999999954|1156      |1              |
|53243585a1d6dc2643021fd1853d8905|222776.04999999993|410       |2              |
|4a3ca9315b744ce9f8e9374361493884|200472.9199999995 |1987      |3              |
|fa1c13f2614d7b5c4749cbc52fecda94|194042.02999999985|586       |4              |
|7c67e1448b00f6e969d365cea6b010ab|187923.89000000063|1364      |5              |
|7e93a43ef30c4f03f38b393420bc753a|176431.87000000008|340       |6              |
|da8622b14eb17ae2831f4ac5b9dab84a|160236.57000000085|1551      |7              |
|7a67c85e85bb2ce8582c35f2203ad736|141745.5299999994 |1171      |8              |
|1025f0e2d44d7041d6cf58b6550e0bfa|138968.55000000022|1428      |9              |
|955fee9216a65b617aa5c053178

26/05/26 18:08:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/26 18:08:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/26 18:08:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
